# Лабораторная работа 4 — PyCaret + SketchBoost

**Порядок запуска:**
1. Runtime → Change runtime type → **GPU**
2. Запусти первую ячейку (`pip install`)
3. **Runtime → Restart session**
4. Загрузи файлы через Files (левая панель): `cars_end.csv` и `stud_end_v2.csv`
5. Run all

In [15]:
!pip install -q --pre pycaret py-boost cupy-cuda12x

In [8]:
!nvidia-smi | grep CUDA

| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |


In [9]:
import cupy
print(cupy.__version__)
import py_boost
print(dir(py_boost))

14.0.1
['CUDA_FOUND', 'Callback', 'GradientBoosting', 'Loss', 'Metric', 'ONNXPredictor', 'SketchBoost', 'TLCompiledPredictor', 'TLPredictor', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_logger', '_root_logger', 'callbacks', 'gpu', 'importlib_metadata', 'logging', 'multioutput', 'pb_to_onnx', 'quantization', 'sampling', 'subprocess', 'sys', 'utils', 'warnings']


In [1]:
import pycaret, sys
print("Python:", sys.version)
print("PyCaret:", pycaret.__version__)

from pycaret.regression import RegressionExperiment
print("RegressionExperiment methods:", [m for m in dir(RegressionExperiment()) if not m.startswith('_')])


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyCaret: 4.0.0a8
RegressionExperiment methods: ['X', 'X_test', 'X_train', 'add_metric', 'automl', 'blend_models', 'calibrate_model', 'compare_models', 'create_model', 'describe_setup_params', 'ensemble_model', 'evaluate_model', 'events', 'feature_selection', 'finalize_model', 'fit', 'fold', 'fold_strategy', 'get_config', 'get_leaderboard', 'get_metadata_routing', 'get_metrics', 'get_params', 'interpret_model', 'list_metrics_cards', 'list_models', 'load_experiment', 'load_model', 'log_experiment', 'logger', 'models', 'n_jobs', 'normalize', 'plot_model', 'predict_model', 'preprocess', 'preprocess_pipeline', 'pull', 'remove_metric', 'remove_outliers', 'save_experiment', 'save_model', 'session_id', 'set_config', 'set_params', 'stack_models', 'target', 'task', 'train_size', 'transformation', 'tune_model', 'use_gpu', 'verbose', 'y', 'y_test', 'y_train']


---
## Часть 1. Регрессия — PyCaret (`cars_end.csv`, target: `price_usd`)

In [2]:
import pandas as pd
from pycaret.regression import RegressionExperiment

df_reg = pd.read_csv('cars_end.csv')

X_reg = df_reg.drop(columns=['price_usd'])
y_reg = df_reg['price_usd']

exp_reg = RegressionExperiment(session_id=42, train_size=0.7, verbose=False)
exp_reg.fit(X_reg, y_reg)
exp_reg.models()

,Name,Reference,Turbo
ID,,,
lr,Linear Regression,sklearn.linear_model._base.LinearRegression,True
lasso,Lasso Regression,sklearn.linear_model._coordinate_descent.Lasso,True
ridge,Ridge Regression,sklearn.linear_model._ridge.Ridge,True
en,Elastic Net,sklearn.linear_model._coordinate_descent.Elast...,True
lar,Least Angle Regression,sklearn.linear_model._least_angle.Lars,True
llar,Lasso Least Angle Regression,sklearn.linear_model._least_angle.LassoLars,True
omp,Orthogonal Matching Pursuit,sklearn.linear_model._omp.OrthogonalMatchingPu...,True
br,Bayesian Ridge,sklearn.linear_model._bayes.BayesianRidge,True
ard,Automatic Relevance Determination,sklearn.linear_model._bayes.ARDRegression,False


In [5]:
dt_reg = exp_reg.create_model('dt')
display(exp_reg.pull())

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold 0,1733.3614,8.717667e+06,2952.5696,0.8061,0.5027,0.0
Fold 1,1764.9788,8.501850e+06,2915.7932,0.8045,0.5572,0.0
Fold 2,1747.7768,9.192358e+06,3031.8901,0.7831,0.5141,0.0
Fold 3,1669.5051,7.439247e+06,2727.4983,0.8172,0.5257,0.0
Fold 4,1678.4499,7.490390e+06,2736.8577,0.7995,0.5384,0.0
Fold 5,1672.8667,7.303108e+06,2702.4263,0.8211,0.5112,0.0
Fold 6,1774.0473,8.639824e+06,2939.3578,0.7767,0.5287,0.0
Fold 7,1737.2179,8.599665e+06,2932.5185,0.8099,0.5140,0.0
Fold 8,1652.9934,7.059366e+06,2656.9467,0.8172,0.5170,0.0
Fold 9,1723.5986,8.569610e+06,2927.3896,0.8036,0.5298,0.0


In [6]:
tuned_dt_reg = exp_reg.tune_model('dt')
display(exp_reg.pull())

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold 0,1452.7583,6.647613e+06,2578.2964,0.8522,0.4223,0.0
Fold 1,1478.8339,6.192788e+06,2488.5314,0.8576,0.4495,0.0
Fold 2,1447.5140,6.283115e+06,2506.6142,0.8517,0.4258,0.0
Fold 3,1388.6442,5.074125e+06,2252.5818,0.8753,0.4431,0.0
Fold 4,1452.1035,5.844795e+06,2417.6011,0.8436,0.4354,0.0
Fold 5,1390.6798,5.272065e+06,2296.0978,0.8709,0.4123,0.0
Fold 6,1456.7270,6.099234e+06,2469.6628,0.8424,0.4429,0.0
Fold 7,1480.3995,6.093872e+06,2468.5769,0.8653,0.4290,0.0
Fold 8,1444.0400,6.049152e+06,2459.5024,0.8433,0.4267,0.0
Fold 9,1419.8346,5.638053e+06,2374.4584,0.8708,0.4262,0.0


---
## Часть 2. Регрессия — SketchBoost (`cars_end.csv`)

In [10]:
import numpy as np
from py_boost import SketchBoost
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

df_sk = pd.read_csv('cars_end.csv')
X_sk = df_sk.drop(columns=['price_usd']).values
y_sk = df_sk['price_usd'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_sk, y_sk, test_size=0.3, random_state=42)

sketch_reg = SketchBoost('mse', ntrees=500, lr=0.05, verbose=100)
sketch_reg.fit(X_tr, y_tr)

for split, X, y in [('Train', X_tr, y_tr), ('Test', X_te, y_te)]:
    pred = sketch_reg.predict(X).ravel()
    print(f"{split}: R²={r2_score(y, pred):.4f}  MAE={mean_absolute_error(y, pred):.2f}")


[06:30:43] Stdout logging level is INFO.


INFO:py_boost.callbacks.callback:Stdout logging level is INFO.


[06:30:43] GDBT train starts. Max iter 500, early stopping rounds 100


INFO:py_boost.callbacks.callback:GDBT train starts. Max iter 500, early stopping rounds 100


[06:30:52] Iter 0; 


...

[06:30:57] Iter 400; 


INFO:py_boost.callbacks.callback:Iter 400; 


[06:30:58] Iter 499; 


INFO:py_boost.callbacks.callback:Iter 499; 


Train: R²=0.9473  MAE=990.19
Test: R²=0.9090  MAE=1175.60


---
## Часть 3. Классификация — PyCaret (`stud_end_v2.csv`, target: `Target`)

In [11]:
from pycaret.classification import ClassificationExperiment

df_cls = pd.read_csv('stud_end_v2.csv')

X_cls = df_cls.drop(columns=['Target'])
y_cls = df_cls['Target']

exp_cls = ClassificationExperiment(session_id=42, train_size=0.7, verbose=False)
exp_cls.fit(X_cls, y_cls)
exp_cls.models()

,Name,Reference,Turbo
ID,,,
lr,Logistic Regression,sklearn.linear_model._logistic.LogisticRegression,True
knn,K Neighbors Classifier,sklearn.neighbors._classification.KNeighborsCl...,True
nb,Naive Bayes,sklearn.naive_bayes.GaussianNB,True
dt,Decision Tree Classifier,sklearn.tree._classes.DecisionTreeClassifier,True
svm,SVM - Linear Kernel,sklearn.linear_model._stochastic_gradient.SGDC...,True
rbfsvm,SVM - Radial Kernel,sklearn.svm._classes.SVC,False
gpc,Gaussian Process Classifier,sklearn.gaussian_process._gpc.GaussianProcessC...,False
mlp,MLP Classifier,sklearn.neural_network._multilayer_perceptron....,False
ridge,Ridge Classifier,sklearn.linear_model._ridge.RidgeClassifier,True


In [12]:
dt_cls = exp_cls.create_model('dt')
display(exp_reg.pull())

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold 0,1452.7583,6.647613e+06,2578.2964,0.8522,0.4223,0.0
Fold 1,1478.8339,6.192788e+06,2488.5314,0.8576,0.4495,0.0
Fold 2,1447.5140,6.283115e+06,2506.6142,0.8517,0.4258,0.0
Fold 3,1388.6442,5.074125e+06,2252.5818,0.8753,0.4431,0.0
Fold 4,1452.1035,5.844795e+06,2417.6011,0.8436,0.4354,0.0
Fold 5,1390.6798,5.272065e+06,2296.0978,0.8709,0.4123,0.0
Fold 6,1456.7270,6.099234e+06,2469.6628,0.8424,0.4429,0.0
Fold 7,1480.3995,6.093872e+06,2468.5769,0.8653,0.4290,0.0
Fold 8,1444.0400,6.049152e+06,2459.5024,0.8433,0.4267,0.0
Fold 9,1419.8346,5.638053e+06,2374.4584,0.8708,0.4262,0.0


In [13]:
tuned_dt_cls = exp_cls.tune_model('dt')
display(exp_reg.pull())

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold 0,1452.7583,6.647613e+06,2578.2964,0.8522,0.4223,0.0
Fold 1,1478.8339,6.192788e+06,2488.5314,0.8576,0.4495,0.0
Fold 2,1447.5140,6.283115e+06,2506.6142,0.8517,0.4258,0.0
Fold 3,1388.6442,5.074125e+06,2252.5818,0.8753,0.4431,0.0
Fold 4,1452.1035,5.844795e+06,2417.6011,0.8436,0.4354,0.0
Fold 5,1390.6798,5.272065e+06,2296.0978,0.8709,0.4123,0.0
Fold 6,1456.7270,6.099234e+06,2469.6628,0.8424,0.4429,0.0
Fold 7,1480.3995,6.093872e+06,2468.5769,0.8653,0.4290,0.0
Fold 8,1444.0400,6.049152e+06,2459.5024,0.8433,0.4267,0.0
Fold 9,1419.8346,5.638053e+06,2374.4584,0.8708,0.4262,0.0


---
## Часть 4. Классификация — SketchBoost (`stud_end_v2.csv`)

In [14]:
from py_boost import SketchBoost
from sklearn.metrics import f1_score, accuracy_score

df_sk2 = pd.read_csv('stud_end_v2.csv')
X_sk2 = df_sk2.drop(columns=['Target']).values.astype(float)
y_sk2 = df_sk2['Target'].values

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_sk2, y_sk2, test_size=0.3, random_state=42, stratify=y_sk2
)

sketch_cls = SketchBoost('crossentropy', ntrees=500, lr=0.05, verbose=100)
sketch_cls.fit(X_tr2, y_tr2)

for split, X, y in [('Train', X_tr2, y_tr2), ('Test', X_te2, y_te2)]:
    pred = np.argmax(sketch_cls.predict(X), axis=1)
    print(f"{split}: F1={f1_score(y, pred, average='weighted'):.4f}  Acc={accuracy_score(y, pred):.4f}")

[06:31:51] Stdout logging level is INFO.


INFO:py_boost.callbacks.callback:Stdout logging level is INFO.


[06:31:51] GDBT train starts. Max iter 500, early stopping rounds 100


INFO:py_boost.callbacks.callback:GDBT train starts. Max iter 500, early stopping rounds 100


[06:31:54] Iter 0; 


...

[06:32:01] Iter 400; 


INFO:py_boost.callbacks.callback:Iter 400; 


[06:32:03] Iter 499; 


INFO:py_boost.callbacks.callback:Iter 499; 


Train: F1=0.9883  Acc=0.9884
Test: F1=0.7699  Acc=0.7726
